In [1]:
!pip install pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00


In [2]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   TS-FORECASTING KAGGLE — v40 (LGB-GPU + TabNet + N-HiTS Features)           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  CHIẾN THUẬT (giữ nguyên nền tảng v38/v39):                                  ║
║    ✓ VAL_THRESHOLD = 3500                                                    ║
║    ✓ Per-horizon models (h=1,3,10,25)                                        ║
║    ✓ 2-stage: mdl_val → best_iter → mdl_full                                 ║
║    ✓ DiffRoC features (diff3/5, roc5/10, accel)                              ║
║    ✓ Polars FE                                                               ║
║                                                                              ║
║  NÂNG CẤP v40 (Từ bài báo N-HiTS: arXiv:2201.12886):                         ║
║    + Thay thế N-BEATS features bằng N-HiTS features:                         ║
║        - Multi-rate Sampling: MaxPool & AvgPool ở các window [3, 7, 14, 25]  ║
║        - Hierarchical Interpolation: Độ lệch giữa các pooling scales         ║
║        - Residuals: Biến động giá trị so với tín hiệu đã downsample          ║
║    + Giữ nguyên LightGBM params của v39                                      ║
║    + Adaptive blend: LGB*w + TabNet*(1-w) theo val scores                    ║
║                                                                              ║
║  OUTPUT: submission.csv + diagnostic plots                                   ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import gc
import os
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import lightgbm as lgb
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
T0 = time.time()

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_GPU = DEVICE == 'cuda'

if USE_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f' GPU: {gpu_name} | VRAM: {gpu_mem:.1f} GB')
else:
    print(' CPU mode — TabNet sẽ chậm; LGB vẫn chạy bình thường')

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG (GIỮ NGUYÊN TỪ V39)
# ══════════════════════════════════════════════════════════════════════════════
TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH  = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH, TEST_PATH = 'train.parquet', 'test.parquet'

# ── Core strategy ─────────────────────────────────────────────────────────────
VAL_THRESHOLD = 3500
HORIZONS      = [1, 3, 10, 25]
SEEDS         = [42, 2024, 12345, 777, 9999]   
CLIP_Q_LOW    = 0.005
CLIP_Q_HIGH   = 0.995

# ── TabNet flag ───────────────────────────────────────────────────────────────
USE_TABNET    = True   # set False nếu GPU OOM hoặc muốn debug LGB nhanh

# ── LightGBM params (GIỮ NGUYÊN) ──────────────────────────────────────────────
LGB_PARAMS = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.015,
    'n_estimators':      5000,
    'num_leaves':        90,
    'min_child_samples': 200,
    'feature_fraction':  0.65,
    'bagging_fraction':  0.75,
    'bagging_freq':      5,
    'lambda_l1':         0.1,
    'lambda_l2':         10.0,
    'verbosity':         -1,
    'n_jobs':            -1,
}
if USE_GPU:
    LGB_PARAMS['device']     = 'gpu'
    LGB_PARAMS['gpu_use_dp'] = False   # float32 GPU → nhanh hơn float64
    print('   LightGBM: device=gpu mode enabled')

# ── TabNet hyperparameters ────────────────────────────────────────────────────
TABNET_MODEL_PARAMS = dict(
    n_d=32, n_a=32, n_steps=5, gamma=1.5,
    n_independent=2, n_shared=2,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={'lr': 0.02, 'weight_decay': 1e-5},
    scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
    scheduler_params={'mode': 'min', 'patience': 3, 'factor': 0.5},
    mask_type='sparsemax',
    verbose=0,
    device_name=DEVICE,
)
TABNET_FIT_PARAMS = dict(
    max_epochs=50,
    patience=8,
    batch_size=16384,
    virtual_batch_size=512,
    num_workers=0,
    drop_last=False,
)

def _clip01(x):
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    """Competition metric."""
    y_target = np.asarray(y_target, dtype=np.float64)
    y_pred   = np.asarray(y_pred,   dtype=np.float64)
    w        = np.asarray(w,        dtype=np.float64)
    denom    = np.sum(w * y_target**2)
    if denom <= 0:
        return 0.0
    ratio = np.sum(w * (y_target - y_pred)**2) / denom
    return float(np.sqrt(1.0 - _clip01(ratio)))

def log_ram(label=''):
    try:
        import psutil
        mb = psutil.Process(os.getpid()).memory_info().rss / 1024**2
        print(f'   [RAM {label}]: {mb:,.0f} MB ({mb/1024:.1f} GB)')
    except ImportError:
        pass

def build_features_polars(train_path, test_path):
    """
    FE v40 = Lags + DiffRoC + Target Enc + N-HiTS Multi-rate Sampling & Hierarchical Interpolation
    """
    print(' Feature Engineering v40 (N-HiTS + DiffRoC)...')

    # ── Đọc & concat ─────────────────────────────────────────────────────────
    df_train = pl.read_parquet(train_path)
    df_test  = pl.read_parquet(test_path)

    df_train = df_train.with_columns(pl.lit(1, dtype=pl.Int8).alias('is_train'))
    df_test  = df_test.with_columns(
        pl.lit(0, dtype=pl.Int8).alias('is_train'),
        pl.lit(None).cast(pl.Float64).alias('y_target'),
        pl.lit(None).cast(pl.Float64).alias('weight'),
    ).select(df_train.columns)

    df = pl.concat([df_train, df_test]).sort(
        ['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    del df_train, df_test
    gc.collect()
    print(f'   concat done: {len(df):,} rows | {df.width} cols')
    log_ram('after concat')

    # ── Target Encoding (fit only on ts ≤ 3500 — KHÔNG LEAK) ────────────────
    fit_df      = df.filter((pl.col('is_train') == 1) &
                            (pl.col('ts_index') <= VAL_THRESHOLD))
    global_mean = float(fit_df.select(pl.col('y_target').mean()).item())

    cat_enc  = fit_df.group_by('sub_category').agg(
        pl.col('y_target').mean().alias('sub_category_enc'))
    code_enc = fit_df.group_by('sub_code').agg(
        pl.col('y_target').mean().alias('sub_code_enc'))

    df = (df.join(cat_enc,  on='sub_category', how='left')
            .with_columns(pl.col('sub_category_enc').fill_null(global_mean)))
    df = (df.join(code_enc, on='sub_code', how='left')
            .with_columns(pl.col('sub_code_enc').fill_null(global_mean)))
    del fit_df, cat_enc, code_enc
    gc.collect()

    exprs = []
    if 'feature_al' in df.columns and 'feature_am' in df.columns:
        exprs += [
            (pl.col('feature_al') - pl.col('feature_am')).alias('d_al_am'),
            (pl.col('feature_al') / (pl.col('feature_am') + 1e-7)).alias('r_al_am'),
        ]
    if 'feature_cg' in df.columns and 'feature_by' in df.columns:
        exprs.append((pl.col('feature_cg') - pl.col('feature_by')).alias('d_cg_by'))
    if exprs:
        df = df.with_columns(exprs)
        exprs.clear()

    cs_cols = ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am']
    for c in cs_cols:
        if c in df.columns:
            exprs.append(
                ((pl.col(c) - pl.col(c).mean().over('ts_index')) /
                 (pl.col(c).std().over('ts_index') + 1e-7)).alias(f'{c}_cs'))

    exprs += [
        (np.sin(2 * np.pi * pl.col('ts_index') / 100.0)).alias('t_sin'),
        (np.cos(2 * np.pi * pl.col('ts_index') / 100.0)).alias('t_cos'),
    ]
    df = df.with_columns(exprs)
    exprs.clear()

    # ── Lag + Rolling + DiffRoC (Base features) ───────────────────────────────
    target_cols = [c for c in
        ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'feature_s']
        if c in df.columns]
    group_cols = ['code', 'sub_code', 'sub_category', 'horizon']

    for c in target_cols:
        for lag in [1, 3, 5, 10, 25]:
            exprs.append(pl.col(c).shift(lag).over(group_cols).alias(f'{c}_lag{lag}'))
        for w in [5, 10, 20, 25]:
            exprs.append(pl.col(c).rolling_mean(w, min_periods=1).over(group_cols)
                            .alias(f'{c}_roll_mean_{w}'))
            exprs.append(pl.col(c).rolling_std(w, min_periods=1).over(group_cols)
                            .alias(f'{c}_roll_std_{w}'))
        exprs.append(pl.col(c).ewm_mean(span=10, min_periods=1, ignore_nulls=True)
                        .over(group_cols).alias(f'{c}_ewm_10'))
        exprs.append(pl.col(c).diff(1).over(group_cols).alias(f'{c}_diff1'))
        exprs.append((pl.col(c).rank() / pl.count()).over('ts_index')
                        .alias(f'{c}_rank'))

        # ── DiffRoC features ──────────────────────────────────────────────────
        exprs.append(pl.col(c).diff(3).over(group_cols)
                        .cast(pl.Float32).alias(f'{c}_diff3'))
        exprs.append(pl.col(c).diff(5).over(group_cols)
                        .cast(pl.Float32).alias(f'{c}_diff5'))
        for k in [5, 10]:
            shifted = pl.col(c).shift(k).over(group_cols)
            exprs.append(((pl.col(c) - shifted) / (shifted.abs() + 1e-7))
                            .clip(-5.0, 5.0).cast(pl.Float32).alias(f'{c}_roc{k}'))
        exprs.append((
            pl.col(c)
            - 2 * pl.col(c).shift(1).over(group_cols)
            + pl.col(c).shift(2).over(group_cols)
        ).cast(pl.Float32).alias(f'{c}_accel'))
        
    df = df.with_columns(exprs)
    exprs.clear()
    print(f'   Pass 1 (lags+rolling+DiffRoC) done: {len(target_cols)} source cols')

    # ── PASS 2: N-HiTS Decomposition Features ─────────────────────────────────
    # N-HiTS Core 1: Multi-rate input sampling (MaxPool / AvgPool over different bands)
    # N-HiTS Core 2: Hierarchical interpolation (Differences between subsampled signals)
    
    rates = [3, 7, 14, 25]  # Tương ứng với các tần số/chu kỳ khác nhau
    
    # BƯỚC 2.1: Tính toán các lớp Pooling trước
    for c in target_cols:
        # Multi-rate Downsampling Features
        for r in rates:
            exprs.append(pl.col(c).rolling_max(r, min_periods=1).over(group_cols)
                         .cast(pl.Float32).alias(f'{c}_nhits_maxp_{r}'))
            exprs.append(pl.col(c).rolling_mean(r, min_periods=1).over(group_cols)
                         .cast(pl.Float32).alias(f'{c}_nhits_avgp_{r}'))

    # Cập nhật DataFrame ngay lập tức để Polars ghi nhận các cột maxp và avgp vừa tạo
    df = df.with_columns(exprs)
    exprs.clear()

    # BƯỚC 2.2: Lấy các lớp Pooling ra tính toán Hierarchical Differences & Residuals
    for c in target_cols:
        # Hierarchical Differences (Interpolation residuals)
        exprs.append((pl.col(f'{c}_nhits_avgp_3') - pl.col(f'{c}_nhits_avgp_14'))
                     .cast(pl.Float32).alias(f'{c}_nhits_interp_short'))
        exprs.append((pl.col(f'{c}_nhits_avgp_7') - pl.col(f'{c}_nhits_avgp_25'))
                     .cast(pl.Float32).alias(f'{c}_nhits_interp_long'))

        # Tín hiệu nội suy phần dư (Residuals từ pooled signals)
        exprs.append((pl.col(c) - pl.col(f'{c}_nhits_maxp_7'))
                     .cast(pl.Float32).alias(f'{c}_nhits_resid_max7'))
        exprs.append((pl.col(c) - pl.col(f'{c}_nhits_avgp_14'))
                     .cast(pl.Float32).alias(f'{c}_nhits_resid_avg14'))

    df = df.with_columns(exprs)
    exprs.clear()
    print(f'   Pass 2 (N-HiTS multi-rate decomposition) done: {len(target_cols)} source cols')
   

    # ── ĐOẠN ĐUÔI NÀY BẮT BUỘC PHẢI CÓ Ở CUỐI HÀM ────────────────────────────
    # ── Ép Float32, fill NaN → 0 
    df = df.with_columns(cs.float().cast(pl.Float32).fill_null(0.0))

    print(f'   Total: {df.width} cols')
    log_ram('before to_pandas')

    # Chuyển đổi về Pandas DataFrame
    df_train_pd = df.filter(pl.col('is_train') == 1).drop('is_train').to_pandas()
    df_test_pd  = (df.filter(pl.col('is_train') == 0)
                     .drop(['is_train', 'y_target', 'weight'])
                     .to_pandas())
    del df
    gc.collect()
    log_ram('FE done')

    # Phải có lệnh return này!
    return df_train_pd, df_test_pd
# ══════════════════════════════════════════════════════════════════════════════
# TABNET TRAINING
# ══════════════════════════════════════════════════════════════════════════════
def train_tabnet(X_fit, y_fit, X_val, y_val, X_all, y_all, X_te, seed=42):
    try:
        from pytorch_tabnet.tab_model import TabNetRegressor
    except ImportError:
        raise RuntimeError('pytorch-tabnet not installed. Run: pip install pytorch-tabnet')

    n_features = X_fit.shape[1]

    def _make_model(s):
        return TabNetRegressor(
            input_dim=n_features,
            output_dim=1,
            seed=s,
            **TABNET_MODEL_PARAMS,
        )

    # Stage 1: find best epoch
    m1 = _make_model(seed)
    m1.fit(
        X_fit, y_fit.reshape(-1, 1),
        eval_set=[(X_val, y_val.reshape(-1, 1))],
        eval_name=['val'],
        eval_metric=['mse'],
        **TABNET_FIT_PARAMS,
    )
    best_epoch   = max(m1.best_epoch, 5)
    val_pred_tab = m1.predict(X_val).flatten()
    del m1
    if USE_GPU:
        torch.cuda.empty_cache()
    gc.collect()

    # Stage 2: retrain on full data
    fit2_params = {**TABNET_FIT_PARAMS, 'max_epochs': best_epoch, 'patience': best_epoch + 1}
    m2 = _make_model(seed)
    m2.fit(
        X_all, y_all.reshape(-1, 1),
        **fit2_params,
    )
    test_pred_tab = m2.predict(X_te).flatten()
    del m2
    if USE_GPU:
        torch.cuda.empty_cache()
    gc.collect()

    return val_pred_tab, test_pred_tab, best_epoch

# ══════════════════════════════════════════════════════════════════════════════
# PER-HORIZON SOLVER (LGB + TabNet + Adaptive Blend)
# ══════════════════════════════════════════════════════════════════════════════
def solve_horizon(horizon, train_df, test_df):
    t_h = time.time()
    print(f"\n{'='*70}\n 🚀 HORIZON {horizon}\n{'='*70}")

    tr = train_df[train_df['horizon'] == horizon].copy()
    te = test_df[test_df['horizon'] == horizon].copy()

    EXCL  = {'id', 'code', 'sub_code', 'sub_category',
             'horizon', 'ts_index', 'weight', 'y_target'}
    feats = [c for c in tr.columns if c not in EXCL]

    n_nhits = sum(1 for c in feats if '_nhits_' in c)
    n_roc    = sum(1 for c in feats if any(x in c for x in ['_roc', '_accel', '_diff3', '_diff5']))
    print(f'Data: train={len(tr):,}, test={len(te):,} | '
          f'Features: {len(feats)} (N-HiTS={n_nhits}, DiffRoC={n_roc})')

    # ── Split ─────────────────────────────────────────────────────────────────
    fit_m = tr['ts_index'] <= VAL_THRESHOLD
    val_m = tr['ts_index'] >  VAL_THRESHOLD

    X_fit = tr.loc[fit_m, feats];   y_fit = tr.loc[fit_m, 'y_target'].values
    w_fit = tr.loc[fit_m, 'weight'].values
    X_val = tr.loc[val_m, feats];   y_val = tr.loc[val_m, 'y_target'].values
    w_val = tr.loc[val_m, 'weight'].values
    X_all = tr[feats];              y_all = tr['y_target'].values
    w_all = tr['weight'].values
    ts_val = tr.loc[val_m, 'ts_index'].values

    # ══ 1. LightGBM (2-stage, N seeds) ═══════════════════════════════════════
    print(f'   [LGB] {len(SEEDS)} seeds ...')
    val_pred_lgb  = np.zeros(len(y_val), dtype=np.float64)
    test_pred_lgb = np.zeros(len(te),    dtype=np.float64)
    best_iters = []
    fi_cum     = np.zeros(len(feats), dtype=np.float64)

    for i, seed in enumerate(SEEDS, 1):
        mdl_val = lgb.LGBMRegressor(**LGB_PARAMS, random_state=seed)
        mdl_val.fit(
            X_fit, y_fit, sample_weight=w_fit,
            eval_set=[(X_val, y_val)], eval_sample_weight=[w_val],
            callbacks=[
                lgb.early_stopping(200, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )
        bi = max(int(mdl_val.best_iteration_ or LGB_PARAMS['n_estimators']), 20)
        best_iters.append(bi)
        val_pred_lgb += mdl_val.predict(X_val) / len(SEEDS)
        del mdl_val
        gc.collect()

        mdl_full = lgb.LGBMRegressor(
            **{**LGB_PARAMS, 'n_estimators': bi}, random_state=seed)
        mdl_full.fit(
            X_all, y_all, sample_weight=w_all,
            callbacks=[lgb.log_evaluation(period=-1)],
        )
        test_pred_lgb += mdl_full.predict(te[feats]) / len(SEEDS)
        fi_cum        += mdl_full.feature_importances_
        del mdl_full
        gc.collect()

    score_lgb = weighted_rmse_score(y_val, val_pred_lgb, w_val)
    print(f'   LGB: score={score_lgb:.6f} | avg_iter={np.mean(best_iters):.0f}')

    # ══ 2. TabNet (2-stage, 1 seed) ══════════════════════════════════════════
    score_tab      = None
    val_pred_tab   = None
    test_pred_tab  = None

    if USE_TABNET:
        print(f'   [TabNet] device={DEVICE} ...')
        try:
            Xf_np  = X_fit.values.astype(np.float32)
            yf_np  = y_fit.astype(np.float32)
            Xv_np  = X_val.values.astype(np.float32)
            yv_np  = y_val.astype(np.float32)
            Xa_np  = X_all.values.astype(np.float32)
            ya_np  = y_all.astype(np.float32)
            Xte_np = te[feats].values.astype(np.float32)

            val_pred_tab, test_pred_tab, best_ep = train_tabnet(
                Xf_np, yf_np, Xv_np, yv_np, Xa_np, ya_np, Xte_np, seed=42)
            score_tab = weighted_rmse_score(y_val, val_pred_tab, w_val)
            print(f'   TabNet: score={score_tab:.6f} | best_epoch={best_ep}')

            del Xf_np, yf_np, Xv_np, yv_np, Xa_np, ya_np, Xte_np
            gc.collect()
        except Exception as e:
            print(f'   ⚠️ TabNet failed: {e!r}')
            print('   → Falling back to LGB-only')
            val_pred_tab  = None
            test_pred_tab = None

    # ══ 3. Adaptive Blend ════════════════════════════════════════════════════
    if val_pred_tab is not None and score_tab is not None:
        err_lgb = max(1e-9, 1.0 - score_lgb)
        err_tab = max(1e-9, 1.0 - score_tab)
        w_lgb   = err_tab / (err_lgb + err_tab)
        w_lgb = max(w_lgb, 0.50)
        w_tab = 1.0 - w_lgb

        val_pred  = w_lgb * val_pred_lgb  + w_tab * val_pred_tab
        test_pred = w_lgb * test_pred_lgb + w_tab * test_pred_tab
        score     = weighted_rmse_score(y_val, val_pred, w_val)
        print(f'   Blend: LGB×{w_lgb:.3f} + TabNet×{w_tab:.3f} → score={score:.6f}')
    else:
        val_pred  = val_pred_lgb
        test_pred = test_pred_lgb
        score     = score_lgb
        w_lgb, w_tab = 1.0, 0.0

    # ── Clip predictions ──────────────────────────────────────────────────────
    q_lo, q_hi = np.quantile(y_fit, [CLIP_Q_LOW, CLIP_Q_HIGH])
    test_clip   = np.clip(test_pred, q_lo, q_hi)

    print(f'💡 h={horizon} | blend={score:.6f} (LGB={score_lgb:.4f}'
          + (f', Tab={score_tab:.4f}' if score_tab else '')
          + f') | {(time.time()-t_h)/60:.1f} min')

    fi_df = (pd.DataFrame({'feature': feats, 'importance': fi_cum / len(SEEDS)})
                .sort_values('importance', ascending=False)
                .reset_index(drop=True))

    out = {
        'horizon':        horizon,
        'id_test':        te['id'].values,
        'test_pred_raw':  test_pred,
        'test_pred_clip': test_clip,
        'id_val':         tr.loc[val_m, 'id'].values,
        'y_val':          y_val,
        'w_val':          w_val,
        'val_pred':       val_pred,
        'ts_val':         ts_val,
        'score_local':    score,
        'score_lgb':      score_lgb,
        'score_tab':      score_tab,
        'w_lgb':          w_lgb,
        'w_tab':          w_tab,
        'feature_imp':    fi_df,
    }

    del tr, te, X_fit, y_fit, w_fit, X_val, y_val, w_val, X_all, y_all, w_all
    del val_pred, test_pred, test_clip, fi_cum, fi_df, ts_val
    gc.collect()

    return out

# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC PLOTS (Cập nhật cho N-HiTS)
# ══════════════════════════════════════════════════════════════════════════════
def create_diagnostic_plots(all_outputs, oof):
    print('\n📊 Generating diagnostic plots...')
    DARK = '#111827'
    HC   = {1: '#2196F3', 3: '#4CAF50', 10: '#FF9800', 25: '#E91E63'}
    V38_SCORES = {1: 0.0725, 3: 0.1420, 10: 0.2282, 25: 0.2730} 

    # ── Plot 1: Score comparison v38 vs v40 ───────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 4))
    fig.patch.set_facecolor(DARK)
    ax.set_facecolor('#1f2937')

    h_list = sorted([o['horizon'] for o in all_outputs])
    s_v40  = [next(o['score_local'] for o in all_outputs if o['horizon'] == h) for h in h_list]
    s_v38  = [V38_SCORES[h] for h in h_list]
    x      = np.arange(len(h_list))

    ax.bar(x - 0.2, s_v38, 0.35, label='v38 local', color='#4B5563', edgecolor='#6B7280')
    ax.bar(x + 0.2, s_v40, 0.35, label='v40 (LGB+TabNet+N-HiTS)',
           color=[HC[h] for h in h_list], edgecolor='#9CA3AF')
    for i, (s38, s40, h) in enumerate(zip(s_v38, s_v40, h_list)):
        d = s40 - s38
        ax.text(i + 0.2, s40 + 0.003, f'{"+" if d>=0 else ""}{d:.4f}',
                ha='center', va='bottom', fontsize=9,
                color='#4ade80' if d >= 0 else '#f87171')

    ax.set_xticks(x)
    ax.set_xticklabels([f'h={h}' for h in h_list], color='#D1D5DB')
    ax.tick_params(colors='#9CA3AF')
    ax.set_ylabel('Local Score', color='#D1D5DB')
    ax.set_title('Score: v38 vs v40 (N-HiTS Features)', color='#F9FAFB', pad=10)
    ax.legend(fontsize=9, facecolor='#374151', labelcolor='#D1D5DB', edgecolor='#4B5563')
    for sp in ax.spines.values():
        sp.set_edgecolor('#374151')
    plt.tight_layout()
    plt.savefig('diag1_scores.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close()
    print('   ✓ diag1_scores.png')

    # ── Plot 2: Blend weights per horizon ─────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
    fig.patch.set_facecolor(DARK)
    for ax, out in zip(axes, sorted(all_outputs, key=lambda o: o['horizon'])):
        h = out['horizon']
        ax.set_facecolor('#1f2937')
        labels  = ['LGB', 'TabNet']
        weights = [out['w_lgb'], out['w_tab']]
        colors  = [HC[h], '#A78BFA']
        bars = ax.bar(labels, weights, color=colors, edgecolor='#374151')
        for bar, val in zip(bars, weights):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', color='#D1D5DB', fontsize=10)
        scores_txt = f'LGB={out["score_lgb"]:.4f}'
        if out['score_tab']:
            scores_txt += f'\nTab={out["score_tab"]:.4f}'
        ax.set_title(f'h={h}  blend\n{scores_txt}', color='#F9FAFB', fontsize=9)
        ax.set_ylim(0, 1.15)
        ax.tick_params(colors='#6B7280')
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')
    plt.suptitle('Adaptive Blend Weights (LGB vs TabNet) per Horizon',
                 color='#D1D5DB', fontsize=11, y=1.04)
    plt.tight_layout()
    plt.savefig('diag2_blend.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close()
    print('   ✓ diag2_blend.png')

    # ── Plot 3: Temporal residual ──────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.patch.set_facecolor(DARK)
    for ax, out in zip(axes, sorted(all_outputs, key=lambda o: o['horizon'])):
        h, ts = out['horizon'], out['ts_val']
        ax.set_facecolor('#1f2937')
        res  = out['y_val'] - out['val_pred']
        bins = np.percentile(ts, np.linspace(0, 100, 51))
        bx, by, be = [], [], []
        for j in range(len(bins)-1):
            m = (ts >= bins[j]) & (ts < bins[j+1])
            if m.sum() > 20:
                bx.append((bins[j]+bins[j+1])/2)
                by.append(np.mean(res[m]))
                be.append(np.std(res[m]) / np.sqrt(m.sum()))
        bx, by, be = map(np.array, [bx, by, be])
        ax.plot(bx, by, color=HC[h], lw=1.5)
        ax.fill_between(bx, by-be, by+be, alpha=0.2, color=HC[h])
        ax.axhline(0, color='white', ls='--', lw=0.8, alpha=0.5)
        ax.set_title(f'h={h} temporal residual', color='#F9FAFB', fontsize=10)
        ax.set_xlabel('ts_index', color='#9CA3AF', fontsize=8)
        ax.set_ylabel('mean residual', color='#9CA3AF', fontsize=8)
        ax.tick_params(colors='#6B7280', labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')
    plt.suptitle('Temporal Residual — Phẳng = tốt | Spike = regime shift',
                 color='#D1D5DB', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('diag3_residual.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close()
    print('   ✓ diag3_residual.png')

    # ── Plot 4: Feature importance (top 20, highlight N-HiTS + DiffRoC) ───────
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.patch.set_facecolor(DARK)
    for ax, out in zip(axes.flatten(), sorted(all_outputs, key=lambda o: o['horizon'])):
        h  = out['horizon']
        fi = out['feature_imp'].head(20)
        ax.set_facecolor('#1f2937')
        colors = []
        for feat in fi['feature']:
            if '_nhits_' in feat:
                colors.append('#10B981')   # green = N-HiTS new
            elif any(x in feat for x in ['_diff3','_diff5','_roc','_accel']):
                colors.append('#F59E0B')   # amber = DiffRoC v38
            else:
                colors.append(HC[h])       # blue/etc = original
        ax.barh(range(len(fi)), fi['importance'], color=colors, edgecolor='#374151')
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi['feature'], fontsize=7, color='#D1D5DB')
        ax.invert_yaxis()
        ax.set_title(f'h={h} Feature Importance Top 20', color='#F9FAFB', fontsize=11)
        ax.set_xlabel('importance', color='#9CA3AF', fontsize=8)
        ax.tick_params(colors='#6B7280', labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')
        legend_elems = [
            Patch(facecolor='#10B981', label='N-HiTS (new v40)'),
            Patch(facecolor='#F59E0B', label='DiffRoC (v38)'),
            Patch(facecolor=HC[h],     label='Original features'),
        ]
        ax.legend(handles=legend_elems, fontsize=8, facecolor='#374151',
                  labelcolor='#D1D5DB', edgecolor='#4B5563', loc='lower right')
    plt.suptitle(
        'Feature Importance — Xanh=N-HiTS mới | Vàng=DiffRoC v38 | Màu=Original',
        color='#D1D5DB', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('diag4_feature_imp.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close()
    print('   ✓ diag4_feature_imp.png')


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':

    # Phase 1: FE
    df_train_all, df_test_all = build_features_polars(TRAIN_PATH, TEST_PATH)
    log_ram('after FE')

    # Phase 2: Train per horizon
    all_outputs = []
    for h in HORIZONS:
        all_outputs.append(solve_horizon(h, df_train_all, df_test_all))

    # Phase 3: Tổng hợp submission
    sub_raw_parts, sub_clip_parts, oof_parts = [], [], []
    for out in all_outputs:
        sub_raw_parts.append(
            pd.DataFrame({'id': out['id_test'], 'prediction': out['test_pred_raw']}))
        sub_clip_parts.append(
            pd.DataFrame({'id': out['id_test'], 'prediction': out['test_pred_clip']}))
        oof_parts.append(pd.DataFrame({
            'id':      out['id_val'],
            'y_true':  out['y_val'],
            'y_pred':  out['val_pred'],
            'w':       out['w_val'],
            'horizon': out['horizon'],
        }))

    sub_raw  = pd.concat(sub_raw_parts,  ignore_index=True)
    sub_clip = pd.concat(sub_clip_parts, ignore_index=True)
    oof      = pd.concat(oof_parts,      ignore_index=True)
    del sub_raw_parts, sub_clip_parts, oof_parts
    gc.collect()

    # Giữ thứ tự test gốc
    test_order = pd.read_parquet(TEST_PATH, columns=['id'])
    sub_clip   = test_order.merge(sub_clip, on='id', how='left').fillna(0)
    sub_raw    = test_order.merge(sub_raw,  on='id', how='left').fillna(0)
    del test_order

    agg_local = weighted_rmse_score(
        oof['y_true'].values, oof['y_pred'].values, oof['w'].values)

    # Phase 4: Diagnostic plots
    create_diagnostic_plots(all_outputs, oof)

    # Lưu files
    sub_clip.to_csv('submission.csv',        index=False)
    sub_raw.to_csv( 'submission_noclip.csv', index=False)
    oof.to_csv(     'oof_v40.csv',           index=False)

    # ── Final report ─────────────────────────────────────────────────────────
    V38_REF = {1: 0.0725, 3: 0.1420, 10: 0.2282, 25: 0.2730}
    print(f"\n{'='*70}")
    print(f'🏆 TỔNG KẾT v40 (LGB-GPU + TabNet + N-HiTS)')
    print(f'   Seeds: {len(SEEDS)} | GPU: {USE_GPU} | TabNet: {USE_TABNET}')
    print(f"{'─'*70}")
    print(f'  {"horizon":>8} | {"v38":>8} | {"v40":>8} | {"delta":>8} | '
          f'{"blend_w":>14}')
    print(f'  {"─"*8}-+-{"─"*8}-+-{"─"*8}-+-{"─"*8}-+-{"─"*14}')
    for out in sorted(all_outputs, key=lambda o: o['horizon']):
        h   = out['horizon']
        s40 = out['score_local']
        s38 = V38_REF[h]
        d   = s40 - s38
        tag = '↑' if d > 0 else '↓'
        blend_str = f'LGB×{out["w_lgb"]:.2f}+Tab×{out["w_tab"]:.2f}'
        print(f'  h={h:>6} | {s38:>8.4f} | {s40:>8.4f} | '
              f'{tag}{abs(d):.4f} | {blend_str}')
    print(f"{'─'*70}")
    print(f'🔥 Aggregate Local Score : {agg_local:.6f}')
    print(f"⏱️  Tổng thời gian : {(time.time()-T0)/60:.1f} phút")
    print(f"{'='*70}")

    print("""
CHECKLIST v40 (N-HiTS Eval):

  [N-HiTS FEATURES CÓ HIỆU QUẢ HƠN N-BEATS?]
  → Kiểm tra file `diag4_feature_imp.png`: các feature `_nhits_maxp_`, `_nhits_interp_` 
    có nằm trong Top 20 Feature Importance không?
  → So sánh `Aggregate Local Score` giữa v39 (N-BEATS) và v40 (N-HiTS). 
    Do N-HiTS mạnh hơn trong long-horizon, hãy đặc biệt chú ý score ở h=10 và h=25.

  [TABNET CÓ GIÚP BLEND?]
  → N-HiTS features thường phối hợp tốt hơn với Deep Learning (TabNet) thay vì 
    Gradient Boosting (LightGBM). Nếu `w_tab` tăng ở v40 so với v39, nghĩa là N-HiTS 
    đã làm cho TabNet học tốt hơn.
""")

 CPU mode — TabNet sẽ chậm; LGB vẫn chạy bình thường
 Feature Engineering v40 (N-HiTS + DiffRoC)...
   concat done: 6,784,521 rows | 95 cols
   [RAM after concat]: 15,999 MB (15.6 GB)
   Pass 1 (lags+rolling+DiffRoC) done: 5 source cols
   Pass 2 (N-HiTS multi-rate decomposition) done: 5 source cols
   Total: 272 cols
   [RAM before to_pandas]: 26,846 MB (26.2 GB)
   [RAM FE done]: 29,505 MB (28.8 GB)
   [RAM after FE]: 29,505 MB (28.8 GB)

 🚀 HORIZON 1
Data: train=1,394,653, test=379,617 | Features: 263 (N-HiTS=60, DiffRoC=25)
   [LGB] 5 seeds ...
   LGB: score=0.077978 | avg_iter=369
   [TabNet] device=cpu ...
   ⚠️ TabNet failed: TypeError("ReduceLROnPlateau.step() missing 1 required positional argument: 'metrics'")
   → Falling back to LGB-only
💡 h=1 | blend=0.077978 (LGB=0.0780) | 44.3 min

 🚀 HORIZON 3
Data: train=1,385,816, test=376,558 | Features: 263 (N-HiTS=60, DiffRoC=25)
   [LGB] 5 seeds ...
   LGB: score=0.125680 | avg_iter=817
   [TabNet] device=cpu ...
   ⚠️ TabNet faile

In [3]:
import os
print('Files saved:')
for f in [
    'submission.csv', 'submission_noclip.csv', 'oof_v39.csv',
    'diag1_scores.png', 'diag2_blend.png',
    'diag3_residual.png', 'diag4_feature_imp.png',
]:
    size   = os.path.getsize(f) // 1024 if os.path.exists(f) else 0
    status = f'{size} KB' if size else '  MISSING'
    print(f'  {f}: {status}')
print('\n submission.csv sẵn sàng nộp!')

Files saved:
  submission.csv: 84502 KB
  submission_noclip.csv: 84502 KB
  oof_v39.csv:   MISSING
  diag1_scores.png: 33 KB
  diag2_blend.png: 35 KB
  diag3_residual.png: 199 KB
  diag4_feature_imp.png: 175 KB

 submission.csv sẵn sàng nộp!
